# YOLOv8n + DSC + MFE

Paper-inspired model implementation and 50-epoch training.

In [20]:
!pip install -q "ultralytics==8.4.173"

import os
import zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4


In [21]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/RDD_Project"
ZIP_PATH = f"{PROJECT}/dataset/rdd.v1i.yolov8.zip"
EXTRACT_PATH = "/content/rdd"

os.makedirs(f"{PROJECT}/results", exist_ok=True)
os.makedirs(f"{PROJECT}/checkpoints", exist_ok=True)
os.makedirs(f"{PROJECT}/outputs", exist_ok=True)
os.makedirs(f"{PROJECT}/notebooks", exist_ok=True)

print("ZIP exists:", os.path.exists(ZIP_PATH))

if not os.path.exists("/content/rdd/data.yaml"):
    os.makedirs(EXTRACT_PATH, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_PATH)

    print("Dataset extracted.")

DATA_YAML = "/content/rdd/data.yaml"

print("data.yaml exists:", os.path.exists(DATA_YAML))
print("\nDataset configuration:\n")
print(open(DATA_YAML).read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP exists: True
data.yaml exists: True

Dataset configuration:

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['0', '1', '2', '3']

roboflow:
  workspace: rdd-u23ar
  project: rdd-59irl
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/rdd-u23ar/rdd-59irl/dataset/1


In [24]:
class DirectionalStripConv(nn.Module):

    def __init__(self, c1, c2, k=7):
        super().__init__()

        hidden = max(c2 // 2, 1)

        self.compress = nn.Conv2d(
            c1, hidden, 1, 1, 0, bias=False
        )
        self.bn1 = nn.BatchNorm2d(hidden)

        self.h_conv = nn.Conv2d(
            hidden, hidden,
            kernel_size=(1, k),
            padding=(0, k // 2),
            groups=hidden,
            bias=False
        )

        self.v_conv = nn.Conv2d(
            hidden, hidden,
            kernel_size=(k, 1),
            padding=(k // 2, 0),
            groups=hidden,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(hidden)

        self.aggregate = nn.Conv2d(
            hidden, c2, 3,
            padding=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

        self.shortcut = (
            nn.Identity()
            if c1 == c2
            else nn.Conv2d(c1, c2, 1, bias=False)
        )

    def forward(self, x):

        identity = self.shortcut(x)

        x = self.compress(x)
        x = self.act(self.bn1(x))

        h = self.h_conv(x)
        v = self.v_conv(x)

        x = h + v
        x = self.act(self.bn2(x))

        x = self.aggregate(x)
        x = self.bn3(x)

        return self.act(x + identity)


class MultiLevelFeatureEqualization(nn.Module):

    def __init__(self, channels, out_channels):
        super().__init__()

        self.num_levels = len(channels)

        self.projections = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False)
            for c in channels
        ])

        self.weight_conv = nn.Conv2d(
            out_channels * self.num_levels,
            self.num_levels,
            1
        )

        self.out_conv = nn.Conv2d(
            out_channels,
            out_channels,
            1,
            bias=False
        )

    def forward(self, features):

        target_size = features[0].shape[-2:]

        aligned = []

        for i, feature in enumerate(features):

            if feature.shape[-2:] != target_size:
                feature = F.interpolate(
                    feature,
                    size=target_size,
                    mode="nearest"
                )

            feature = self.projections[i](feature)
            aligned.append(feature)

        concat = torch.cat(aligned, dim=1)

        weights = self.weight_conv(concat)
        weights = torch.softmax(weights, dim=1)

        fused = 0

        for i in range(self.num_levels):
            fused = fused + (
                aligned[i] * weights[:, i:i+1]
            )

        return self.out_conv(fused)


class DSCBlock(nn.Module):

    def __init__(self, original):
        super().__init__()

        self.original = original
        self.dsc = DirectionalStripConv(
            128, 128, k=7
        )

    def forward(self, x):
        return self.dsc(self.original(x))


class MFEFusion(nn.Module):

    def __init__(self):
        super().__init__()

        self.mfe = MultiLevelFeatureEqualization(
            channels=[128, 64],
            out_channels=64
        )

    def forward(self, features):
        return self.mfe(features)


class MFEAdapter(nn.Module):

    def __init__(self, original):
        super().__init__()

        self.adapter = nn.Sequential(
            nn.Conv2d(
                64, 192, 1, bias=False
            ),
            nn.BatchNorm2d(192),
            nn.SiLU()
        )

        self.original = original

    def forward(self, x):
        return self.original(
            self.adapter(x)
        )


# ==========================================
# LOAD YOLOv8n
# ==========================================

model2 = YOLO("yolov8n.pt")
net = model2.model.cuda()


# ==========================================
# DSC
# ==========================================

net.model[12] = DSCBlock(
    net.model[12]
)

net.model[12].f = -1
net.model[12].i = 12


# ==========================================
# MFE
# ==========================================

net.model[14] = MFEFusion()

net.model[14].f = [-1, 4]
net.model[14].i = 14


# ==========================================
# MFE ADAPTER
# ==========================================

net.model[15] = MFEAdapter(
    net.model[15]
)

net.model[15].f = -1
net.model[15].i = 15


# ==========================================
# DETECTION HEAD
# ==========================================

old_detect = net.model[22]

new_detect = Detect(
    nc=4,
    ch=(64, 128, 256)
)

new_detect.stride = old_detect.stride

net.model[22] = new_detect

# CORRECT YOLOv8n connections
net.model[22].f = [15, 18, 21]
net.model[22].i = 22

net.nc = 4

net.names = {
    0: "Longitudinal Crack",
    1: "Transverse Crack",
    2: "Alligator Crack",
    3: "Pothole"
}

net.stride = new_detect.stride

new_detect.bias_init()

net = net.cuda()

print("MODEL 2 CREATED")
print("Detect connections:", net.model[22].f)

MODEL 2 CREATED
Detect connections: [15, 18, 21]


In [26]:
net.eval()

dummy = torch.randn(
    1, 3, 640, 640,
    device="cuda"
)

with torch.no_grad():
    output = net(dummy)

print("FORWARD PASS SUCCESS")
print("Output type:", type(output))

if isinstance(output, dict):
    print("Output keys:", output.keys())
else:
    print("Output shape:", output[0].shape)

FORWARD PASS SUCCESS
Output type: <class 'tuple'>
Output shape: torch.Size([1, 8, 8400])


In [27]:
results2 = model2.train(

    # Dataset
    data=DATA_YAML,

    # Training
    epochs=50,
    imgsz=640,
    batch=16,

    # Optimizer
    optimizer="AdamW",
    lr0=0.0005,
    cos_lr=True,

    # Loss
    box=7.5,
    cls=0.5,

    # Augmentation
    mosaic=1.0,
    close_mosaic=10,
    fliplr=0.5,
    flipud=0.0,
    scale=0.5,
    translate=0.1,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # Warmup
    warmup_epochs=3,

    # Reproducibility
    seed=0,

    # Complete 50-epoch controlled run
    patience=50,

    # GPU
    device=0,
    workers=2,
    amp=True,

    # Checkpoints
    save=True,
    save_period=5,

    # Results
    project=f"{PROJECT}/results",
    name="YOLOv8n_DSC_MFE_50ep_FINAL",
    exist_ok=True,

    plots=True
)

New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/rdd/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937